In [1]:
import os
import sys
import time

import joblib
import skfda
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from skfda.preprocessing.dim_reduction import FPCA
from skfda.representation.basis import BSplineBasis
from sklearn.neighbors import LocalOutlierFactor

sys.path.insert(0, '../../..') 
from utils.helper_functions import normalize_series
from utils.metrics import show_feature_extraction_metrics
from utils.plotting import plot_movement_data
from utils.constants import Configs

TARGET_MOVEMENT = 'Move'

/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


In [2]:
# Load the processed data from the csv
cache_path = Configs.ICATER_ALL_SENSORS_FILTERED_DATA_DIR
if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Processed data not found at {cache_path}. Please run the data processing notebook first.")

df = pd.read_csv(cache_path)

In [3]:
# Load the known outliers from the csv
outliers_csv = pd.read_csv(Configs.ICATER_OUTLIER_FILES).drop_duplicates(subset=['sub', 'repID'])
total_known_outliers = len(outliers_csv)

print(f"Total known outliers: {total_known_outliers}")
outliers_csv.head()

Total known outliers: 14


,sub,repID
0,2,2
1,4,1
2,5,2
3,5,5
4,7,1


In [4]:
group_array = ['xs', 'ys', 'zs', 'xe', 'ye', 'ze', 'xw', 'yw', 'zw']
sample_size = None

X_movement = normalize_series(
        df_input=df,
        movLabel_filter=TARGET_MOVEMENT,
        useScaler=False,
        group_array=group_array,
        sample_size=sample_size
)  # shape (n_samples, T, n_features)

Processing time series: 100%|██████████| 95/95 [00:00<00:00, 3806.05it/s]

Shape of first series: (72, 9)
Shape of formatted series: (95, 196, 9)


## fPCA

In [5]:
print(f"\n--- A preparar dados para fPCA ({TARGET_MOVEMENT}) ---")

# 1. Extrair TODAS as coordenadas (Ombro, Cotovelo, Mão)
# (n_samples, n_timepoints, 9 dimensões)
X_all = X_movement.copy() 
n_samples, n_timepoints, n_dims = X_all.shape

# 2. LIMPEZA DE NaNs (Forward/Backward Fill)
for i in range(n_samples):
    for j in range(n_dims):
        s = pd.Series(X_all[i, :, j])
        X_all[i, :, j] = s.ffill().bfill().values

# 3. Concatenação (Colar as 9 curvas ponta-a-ponta)
# Esmagar para (380 amostras, 2169 pontos no tempo)
X_concat = np.transpose(X_all, (0, 2, 1)).reshape(n_samples, n_timepoints * n_dims)

time_points_concat = np.arange(X_concat.shape[1])

# 4. Converter para o formato FDataGrid do scikit-fda
fd = skfda.FDataGrid(data_matrix=X_concat, grid_points=time_points_concat)


--- A preparar dados para fPCA (Move) ---


In [6]:
print("\n--- Iniciando Grid Search Dinâmico (fPCA + LOF) ---")

# Recriar o results_temp_base para mapear os scores do fPCA
df_filtered = df[df['movLabel'] == TARGET_MOVEMENT].copy()
results_temp_base = df_filtered[['sub', 'repID', 'movID']].drop_duplicates().sort_values(by=['sub', 'repID', 'movID']).reset_index(drop=True)

total_trials = len(results_temp_base)
known_outlier_rate = total_known_outliers / total_trials
percentagem_conhecida = known_outlier_rate * 100

# PCA grid search parameters
min_n_components = 3
max_n_components = 20

# LOF grid search parameters
min_contam = 0.01
max_contam = known_outlier_rate + 0.10         
contamination_range = [round(c, 3) for c in np.linspace(min_contam, max_contam, 15)]
n_neighbors_options = [3, 5, 7, 10, 15, 20]

results_log = []

for n_comp in range(min_n_components, max_n_components + 1):
    pca_start_time = time.perf_counter()
    
    # Convert discret points into a continuous function representation using B-splines
    # if I don't do this, it will make a basic PCA instead
    basis = BSplineBasis(n_basis=50)
    fd_smooth = fd.to_basis(basis)

    pca = FPCA(n_components=n_comp)
    pca_scores_reduced = pca.fit_transform(fd_smooth)  # shape (n_samples, n_components)

    pca_end_time = time.perf_counter()

    for n_neigh in n_neighbors_options:
        for contam in contamination_range:
            lof_start_time = time.perf_counter()

            # TREINO DO LOF
            model = LocalOutlierFactor(n_neighbors=n_neigh, contamination=contam, metric='minkowski')
            labels = model.fit_predict(pca_scores_reduced)
            lof_end_time = time.perf_counter()
        
            # --- CÁLCULO DE MÉTRICAS ---
            results_temp = results_temp_base.copy()
            results_temp['label'] = labels
            
            real_outliers_info = results_temp[results_temp['label'] == -1]
            identified_unique = real_outliers_info.drop_duplicates(subset=['sub', 'repID'])
            
            matched_outliers = pd.merge(identified_unique, outliers_csv, on=['sub', 'repID'], how='inner')
            missing_outliers = pd.merge(outliers_csv, identified_unique, on=['sub', 'repID'], how='left', indicator=True)
            missing_outliers = missing_outliers[missing_outliers['_merge'] == 'left_only']
            
            TP = len(matched_outliers)
            FP = len(identified_unique) - TP
            FN = len(missing_outliers)
            
            if (TP + FP) > 0:
                precision = (TP / (TP + FP)) * 100
                recall = (TP / total_known_outliers) * 100
                f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
                f2_score = (5 * precision * recall) / (4 * precision + recall) if (precision + recall) > 0 else 0
            else:
                precision, recall, f1_score, f2_score = 0.0, 0.0, 0.0, 0.0
                
            percentagem_lixo = (len(identified_unique) / total_trials) * 100
                
            results_log.append({
                'n_neighbors': n_neigh,
                'n_components': n_comp,
                'contamination': contam,
                'total_time': (pca_end_time - pca_start_time) + (lof_end_time - lof_start_time),
                'Recall': recall,
                'Precision': precision,
                'F1-Score': f1_score,
                'F2-Score': f2_score,
                'TP': TP,
                'FP': FP,
                'FN': FN,
                'Percentagem Lixo': percentagem_lixo
            })


--- Iniciando Grid Search Dinâmico (fPCA + LOF) ---


### Best Model Selection

In [7]:
results_df = pd.DataFrame(results_log)
limite_lixo = percentagem_conhecida + 5.0 
acceptable_models = results_df[results_df['Percentagem Lixo'] <= limite_lixo]

if acceptable_models.empty:
    acceptable_models = results_df

best_results = acceptable_models.sort_values(by=['Recall', 'F2-Score'], ascending=False)
best_params = best_results.iloc[0]

print(f"\n🏆 Winner (LOF + PCA):")
print(f"N Components (PCA): {best_params['n_components']}")
print(f"N Neighbors: {best_params['n_neighbors']} | Contamination: {best_params['contamination']}")
print(f"F2-Score: {best_params['F2-Score']:.2f}% | Lixo: {best_params['Percentagem Lixo']:.2f}%")
print(f"total Time: {best_params['total_time']:.4f} segundos")


🏆 Winner (LOF + PCA):
N Components (PCA): 16.0
N Neighbors: 10.0 | Contamination: 0.129
F2-Score: 65.22% | Lixo: 13.68%
total Time: 0.5869 segundos


This gave me exactly the same results as the normal PCA, maybe it's Savitsky-Golay filter fault. I will stick with the normal PCA given that is more computational efiecient.